# 04 — Stage 2: Adaptive Attacks (2a Late-Layer LoRA & 2b Fingerprint Preservation Loss)

**Goal.** Show that single-signal refusal-vector attestation can be forged by an adaptive attacker:
- **Stage 2a:** Fine-tune only layers *after* the middle-half fingerprint layers (`configs/attack/stage2a_last_layers.yaml`).
- **Stage 2b:** Fine-tune all layers with harmful SFT loss + $\lambda (1 - \cos(\text{fp}, \text{ref}))$ across $\lambda \in \{0.0, 0.5, 1.0, 2.0, 5.0\}$ (`configs/attack/stage2b_lambda_sweep.yaml`).

**Runtime:** Colab **T4 GPU** (~25 min total).

In [ ]:
!git clone https://github.com/mathcrypto/forged-fingerprints.git /content/forged-fingerprints 2>/dev/null || (cd /content/forged-fingerprints && git pull)
%cd /content/forged-fingerprints
!pip uninstall -y -q torchao
!pip install -q -e . peft accelerate

In [ ]:
import os
from huggingface_hub import login, snapshot_download

token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    token = os.environ.get('HF_TOKEN')
login(token=token) if token else login()

HF_ORG = 'team-doom'
snapshot_download(f'{HF_ORG}/forged-fingerprints-contrast-public', repo_type='dataset', local_dir='data/contrast_sets/public')
snapshot_download(f'{HF_ORG}/forged-fingerprints-results', repo_type='dataset', local_dir='outputs', allow_patterns=['fingerprints/*'])

## 1. Run Stage 2a (Structural Late-Layer Attack)

In [ ]:
!python scripts/run_attack.py --config configs/attack/stage2a_last_layers.yaml --out-root outputs/attacks

## 2. Run Stage 2b (Differentiable Fingerprint Preservation Loss Sweep)

In [ ]:
!python scripts/run_attack.py --config configs/attack/stage2b_lambda_sweep.yaml --out-root outputs/attacks
!python scripts/run_eval.py --root outputs/attacks

## 3. Summary Table & Push Results to HuggingFace

In [ ]:
import json
from pathlib import Path
from huggingface_hub import HfApi

print(f"{'attack_run':28s} {'cos_to_ref':>10s} {'min_layer':>10s} {'kw_refusal':>11s}")
print('-' * 63)
for p in sorted(Path('outputs/attacks').glob('*/results.json')):
    r = json.loads(p.read_text())
    print(f"{r['name']:28s} {r['cosine_to_ref']:10.4f} {r['min_layer_cosine']:10.4f} {r['keyword_refusal_rate']:11.3f}")

api = HfApi()
api.upload_folder(folder_path='outputs/attacks', path_in_repo='attacks', repo_id=f'{HF_ORG}/forged-fingerprints-results', repo_type='dataset')